# Arrays in PySpark

This notebook demonstrates how to work with array data types in PySpark, including array creation, indexing, and exploding arrays into rows.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import array, col, explode, size, array_contains
from pyspark.sql.functions import array_max, array_min, array_distinct, array_sort

spark = SparkSession.builder \
    .appName('Arrays in PySpark') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Creating DataFrames with Arrays

### Method 1: Direct Creation with Arrays

In [ ]:
# Create DataFrame with array column
data = [
    (1, [1, 2, 3]),
    (2, [4, 5]),
    (3, [6, 7, 8, 9]),
    (4, [10])
]

df = spark.createDataFrame(data, ["id", "values"])

print("DataFrame with Array Column:")
df.show()
df.printSchema()

### Method 2: Creating Arrays from Columns

In [ ]:
# Create DataFrame with separate columns
data2 = [
    (1, "Alice", 85, 90, 88),
    (2, "Bob", 78, 82, 80),
    (3, "Charlie", 92, 88, 95),
    (4, "David", 70, 75, 73)
]

df_scores = spark.createDataFrame(data2, ["id", "name", "math", "science", "english"])

# Create array from existing columns
df_with_array = df_scores.withColumn(
    "all_scores",
    array(col("math"), col("science"), col("english"))
)

print("Array Created from Columns:")
df_with_array.show()
df_with_array.printSchema()

## Accessing Array Elements

### Indexing Arrays (0-based indexing)

In [ ]:
# Access first element (index 0)
print("First Element of Array:")
df.select(col("id"), col("values")[0].alias("first_value")).show()

# Access second element (index 1)
print("Second Element of Array:")
df.select(col("id"), col("values")[1].alias("second_value")).show()

# Access last element (index -1 doesn't work in PySpark, use size-1)
print("Last Element of Array:")
df.select(
    col("id"),
    col("values")[size(col("values")) - 1].alias("last_value")
).show()

### Multiple Element Access

In [ ]:
# Access multiple elements at once
df.select(
    col("id"),
    col("values"),
    col("values")[0].alias("first"),
    col("values")[1].alias("second"),
    col("values")[2].alias("third")
).show()

## Exploding Arrays

### Basic Explode - Convert Array Elements to Rows

In [ ]:
# Explode array into separate rows
print("Original DataFrame:")
df.show()

print("\nAfter Exploding 'values' Array:")
df_exploded = df.withColumn("value", explode(col("values")))
df_exploded.show()

In [ ]:
# Select only specific columns after explode
print("Exploded with Selected Columns:")
df.select(col("id"), explode(col("values")).alias("individual_value")).show()

### Explode with Position (posexplode)

In [ ]:
from pyspark.sql.functions import posexplode

# Explode with position index
print("Explode with Position Index:")
df.select(
    col("id"),
    posexplode(col("values")).alias("position", "value")
).show()

## Array Functions

### Array Size

In [ ]:
# Get size of array
print("Array Sizes:")
df.select(
    col("id"),
    col("values"),
    size(col("values")).alias("array_size")
).show()

### Array Contains

In [ ]:
# Check if array contains a specific value
print("Check if Array Contains 5:")
df.select(
    col("id"),
    col("values"),
    array_contains(col("values"), 5).alias("contains_5")
).show()

# Filter rows where array contains 5
print("\nFilter Rows Where Array Contains 5:")
df.filter(array_contains(col("values"), 5)).show()

### Array Min and Max

In [ ]:
# Get minimum and maximum values from array
print("Min and Max Values in Arrays:")
df.select(
    col("id"),
    col("values"),
    array_min(col("values")).alias("min_value"),
    array_max(col("values")).alias("max_value")
).show()

### Array Sort

In [ ]:
# Create DataFrame with unsorted arrays
data_unsorted = [
    (1, [3, 1, 2]),
    (2, [9, 5, 7]),
    (3, [4, 8, 6, 2])
]

df_unsorted = spark.createDataFrame(data_unsorted, ["id", "values"])

print("Original Arrays:")
df_unsorted.show()

print("Sorted Arrays:")
df_unsorted.select(
    col("id"),
    col("values"),
    array_sort(col("values")).alias("sorted_values")
).show()

### Array Distinct

In [ ]:
# Create DataFrame with duplicate values in arrays
data_duplicates = [
    (1, [1, 2, 2, 3, 3, 3]),
    (2, [4, 4, 5, 5]),
    (3, [6, 7, 6, 8, 7])
]

df_duplicates = spark.createDataFrame(data_duplicates, ["id", "values"])

print("Arrays with Duplicates:")
df_duplicates.show()

print("Arrays with Distinct Values:")
df_duplicates.select(
    col("id"),
    col("values"),
    array_distinct(col("values")).alias("distinct_values")
).show()

## Array Operations

### Array Union (Concatenation)

In [ ]:
from pyspark.sql.functions import array_union, concat

# Create DataFrame with two array columns
data_multi = [
    (1, [1, 2, 3], [4, 5, 6]),
    (2, [7, 8], [9, 10]),
    (3, [11], [12, 13, 14])
]

df_multi = spark.createDataFrame(data_multi, ["id", "array1", "array2"])

print("Two Array Columns:")
df_multi.show()

# Concatenate arrays (includes duplicates)
print("Concatenated Arrays:")
df_multi.select(
    col("id"),
    col("array1"),
    col("array2"),
    concat(col("array1"), col("array2")).alias("concatenated")
).show(truncate=False)

# Union (removes duplicates)
data_overlap = [
    (1, [1, 2, 3], [3, 4, 5]),
    (2, [6, 7], [7, 8, 9])
]

df_overlap = spark.createDataFrame(data_overlap, ["id", "array1", "array2"])

print("\nArray Union (Distinct):")
df_overlap.select(
    col("id"),
    col("array1"),
    col("array2"),
    array_union(col("array1"), col("array2")).alias("union")
).show(truncate=False)

### Array Intersect and Except

In [ ]:
from pyspark.sql.functions import array_intersect, array_except

# Find common elements
print("Array Intersect (Common Elements):")
df_overlap.select(
    col("id"),
    col("array1"),
    col("array2"),
    array_intersect(col("array1"), col("array2")).alias("common")
).show(truncate=False)

# Find elements in array1 but not in array2
print("\nArray Except (Elements in array1 but not in array2):")
df_overlap.select(
    col("id"),
    col("array1"),
    col("array2"),
    array_except(col("array1"), col("array2")).alias("difference")
).show(truncate=False)

## Practical Example: Student Scores

In [ ]:
from pyspark.sql.functions import avg as spark_avg, round as spark_round

# Student scores example
print("Student Scores DataFrame:")
df_with_array.show()

# Calculate statistics on array
print("\nStudent Score Statistics:")
df_stats = df_with_array.select(
    col("id"),
    col("name"),
    col("all_scores"),
    array_min(col("all_scores")).alias("min_score"),
    array_max(col("all_scores")).alias("max_score"),
    size(col("all_scores")).alias("num_subjects")
)
df_stats.show()

# Explode scores and calculate average
print("\nIndividual Scores (Exploded):")
df_individual = df_with_array.select(
    col("id"),
    col("name"),
    explode(col("all_scores")).alias("score")
)
df_individual.show()

# Calculate average per student
print("\nAverage Score per Student:")
df_avg = df_individual.groupBy("id", "name") \
    .agg(spark_round(spark_avg("score"), 2).alias("avg_score"))
df_avg.show()

## Filtering Based on Arrays

In [ ]:
# Filter students with max score > 90
print("Students with Max Score > 90:")
df_with_array.filter(array_max(col("all_scores")) > 90).show()

# Filter students with min score < 75
print("\nStudents with Min Score < 75:")
df_with_array.filter(array_min(col("all_scores")) < 75).show()

# Filter students with more than 2 subjects
print("\nStudents with Array Size = 3:")
df_with_array.filter(size(col("all_scores")) == 3).show()

## Array Aggregations

In [ ]:
from pyspark.sql.functions import collect_list, collect_set

# Create sample data for aggregation
data_agg = [
    ("Math", "John", 85),
    ("Math", "Jane", 92),
    ("Math", "Bob", 78),
    ("Science", "John", 90),
    ("Science", "Jane", 88),
    ("Science", "Bob", 82)
]

df_agg = spark.createDataFrame(data_agg, ["subject", "student", "score"])

print("Original Data:")
df_agg.show()

# Collect scores into arrays per subject
print("\nScores Collected into Arrays (with duplicates):")
df_collected = df_agg.groupBy("subject") \
    .agg(collect_list("score").alias("all_scores"))
df_collected.show(truncate=False)

# Collect distinct students per subject
print("\nDistinct Students per Subject:")
df_students = df_agg.groupBy("subject") \
    .agg(collect_set("student").alias("students"))
df_students.show(truncate=False)

## Key Takeaways

### Array Operations:
- **Creating Arrays**: Use `array()` function or define in data
- **Indexing**: Use `col("array")[index]` (0-based indexing)
- **Exploding**: Use `explode()` to convert array elements to rows
- **Position Explode**: Use `posexplode()` to get index with values

### Array Functions:
- `size()`: Get array length
- `array_contains()`: Check if value exists
- `array_min()` / `array_max()`: Get min/max values
- `array_sort()`: Sort array elements
- `array_distinct()`: Remove duplicates
- `array_union()`: Combine arrays (distinct)
- `array_intersect()`: Common elements
- `array_except()`: Elements in first but not second

### Aggregations:
- `collect_list()`: Aggregate into array (with duplicates)
- `collect_set()`: Aggregate into array (distinct values)

In [ ]:
# Stop Spark Session
spark.stop()